# Data Extraction
First Ill be converting the pdf to text format, since this is what the AI will be seeing as well.  
Then I picked out the relevant passages and crafted out a json representing what the passage means.  
Once this is done, Ill be splitting the json structures to training and validation dataset.

In [1]:
from pathlib import Path
import pymupdf

# Works when launched from the project root or the notebooks directory.
project_root = Path.cwd()
if not (project_root / "datasheets").is_dir():
    project_root = project_root.parent
pdf_dir = project_root / "datasheets"
text_dir = project_root / "data" / "text"
text_dir.mkdir(parents=True, exist_ok=True)
pdfs = sorted(pdf_dir.glob("*.pdf"))
if not pdfs:
    raise FileNotFoundError(f"No PDFs found in {pdf_dir}")

for pdf_path in pdfs:
    with pymupdf.open(pdf_path) as document:
        pages = [
            f"--- PDF page {page.number + 1} ---\n{page.get_text(sort=True)}"
            for page in document
        ]
    text_path = text_dir / f"{pdf_path.stem}.txt"
    text_path.write_text("\n\n".join(pages), encoding="utf-8")
    print(f"{pdf_path.name} -> {text_path.name} ({len(pages)} pages)")


bme280.pdf -> bme280.txt (60 pages)
fdc1004.pdf -> fdc1004.txt (35 pages)
hdc1080.pdf -> hdc1080.txt (29 pages)
ina219.pdf -> ina219.txt (38 pages)
mcp9808.pdf -> mcp9808.txt (53 pages)
mpl3115a2.pdf -> mpl3115a2.txt (51 pages)
opt3001.pdf -> opt3001.txt (44 pages)
sgp30.pdf -> sgp30.txt (19 pages)
sht31.pdf -> sht31.txt (22 pages)
tmag5273.pdf -> tmag5273.txt (61 pages)
tmp117.pdf -> tmp117.txt (50 pages)
veml7700.pdf -> veml7700.txt (15 pages)


## Training, validation and test splits
Preserve the reviewed pilot family assignments. Use GroupShuffleSplit (seed 42) to assign three new families to test, one to validation, and four to training. No family occurs in more than one split.


Combining existing datasheets's json to one:

In [2]:
from pathlib import Path
import hashlib
import json
import sklearn
from sklearn.model_selection import GroupShuffleSplit

project_root = Path.cwd()
if not (project_root / "datasheets").is_dir():
    project_root = project_root.parent

# Preserve the original 100 records' order; append new families alphabetically.
original_families = ["hdc1080", "mpl3115a2", "opt3001", "tmp117"]
source_dir = project_root / "data" / "extracted"
new_families = sorted(p.stem for p in source_dir.glob("*.json")
                      if p.stem not in original_families)
names = original_families + new_families
records, groups = [], []
for family in names:
    examples = json.loads((source_dir / f"{family}.json").read_text(encoding="utf-8"))
    if not isinstance(examples, list) or not examples:
        raise ValueError(f"Expected a nonempty array in {family}.json")
    records.extend(examples)
    groups.extend([family] * len(examples))

combined_path = project_root / "data" / "combined.json"
combined_path.write_text(json.dumps(records, ensure_ascii=False, indent=2) + "\n",
                         encoding="utf-8")
print(f"Combined {len(records)} examples from {len(names)} device families.")


Combined 220 examples from 12 device families.


Splitting data done here:

In [3]:
# Keep the original reviewed pilot assignments fixed.
# Only new families participate in the two seeded random group selections.
seed = 42
original_train = {"hdc1080", "opt3001", "tmp117"}
original_validation = {"mpl3115a2"}
if len(new_families) != 8:
    raise ValueError("This 12-family split expects exactly eight new families; review the split policy if adding more.")

new_indices = [i for i, family in enumerate(groups) if family in new_families]
new_groups = [groups[i] for i in new_indices]
test_splitter = GroupShuffleSplit(n_splits=1, test_size=3, random_state=seed)
development_local, test_local = next(test_splitter.split(new_indices, groups=new_groups))
test_indices = [new_indices[i] for i in test_local]
development_indices = [new_indices[i] for i in development_local]

validation_splitter = GroupShuffleSplit(n_splits=1, test_size=1, random_state=seed)
train_local, validation_local = next(validation_splitter.split(
    development_indices, groups=[groups[i] for i in development_indices]))
train_indices = sorted([i for i, family in enumerate(groups) if family in original_train]
                       + [development_indices[i] for i in train_local])
validation_indices = sorted([i for i, family in enumerate(groups) if family in original_validation]
                            + [development_indices[i] for i in validation_local])
test_indices = sorted(test_indices)
indices = {"train": train_indices, "validation": validation_indices, "test": test_indices}
families = {label: sorted({groups[i] for i in ix}) for label, ix in indices.items()}
for a, b in [("train", "validation"), ("train", "test"), ("validation", "test")]:
    assert set(indices[a]).isdisjoint(indices[b])
    assert set(families[a]).isdisjoint(families[b])
    assert {records[i]["passage"] for i in indices[a]}.isdisjoint(
        records[i]["passage"] for i in indices[b])
assert set().union(*(set(ix) for ix in indices.values())) == set(range(len(records)))
assert len({row["passage"] for row in records}) == len(records)

output_dir = project_root / "data" / "splits"
output_dir.mkdir(parents=True, exist_ok=True)
manifest = {
    "method": "GroupShuffleSplit on new families; original pilot assignments retained",
    "sklearn_version": sklearn.__version__, "seed": seed,
    "new_test_group_count": 3, "new_validation_group_count": 1,
    "combined_sha256": hashlib.sha256(combined_path.read_bytes()).hexdigest(),
    "source_order": names,
    "source_sha256": {family: hashlib.sha256((source_dir / f"{family}.json").read_bytes()).hexdigest()
                      for family in names},
    "index_base": 0,
    "review_status": {family: "user_reviewed" if family in original_families
                      else "assistant_annotated_pending_user_review" for family in names},
    "test_policy": "New families only; keep fixed and do not use for model or hyperparameter selection."
}
for label, ix in indices.items():
    examples = [records[i] for i in ix]
    manifest[f"{label}_families"] = families[label]
    manifest[f"{label}_indices"] = ix
    manifest[f"{label}_count"] = len(examples)
    (output_dir / f"{label}.json").write_text(
        json.dumps(examples, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    negatives = sum(not row["requirements"] for row in examples)
    print(f"{label}: {len(examples)} examples ({negatives} negatives); {', '.join(families[label])}")
(output_dir / "manifest.json").write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print("Saved train.json, validation.json, test.json and manifest.json in data/splits/.")


train: 135 examples (26 negatives); hdc1080, ina219, opt3001, sgp30, tmag5273, tmp117, veml7700
validation: 40 examples (8 negatives); mcp9808, mpl3115a2
test: 45 examples (9 negatives); bme280, fdc1004, sht31
Saved train.json, validation.json, test.json and manifest.json in data/splits/.
